# ConfigMaps, Secrets & Environment

## What's covered

- **The configuration problem** — why baking config into an image is a dead end
- **ConfigMap** — Kubernetes' first-class object for non-sensitive configuration: key/value, key/file, the YAML
- **Three ways to consume a ConfigMap** — `env`, `envFrom`, and as files via a volume mount
- **Update propagation** — env vars are baked at pod start; volume-mounted ConfigMaps live-refresh (with one caveat involving `subPath`)
- **Immutable ConfigMaps and Secrets** — opting out of updates for fewer kubelet wakeups
- **Secrets** — same shape as ConfigMap but for sensitive data: base64 encoding (which is **not** encryption), encryption at rest in `etcd`
- **Built-in Secret types** — `Opaque`, `kubernetes.io/dockerconfigjson`, `kubernetes.io/tls`, and a few others
- **Image pull secrets** — how the kubelet authenticates to private registries
- **The downward API** — exposing pod metadata (name, namespace, IP, labels, resource limits) to the container as env vars or files

By the end of this notebook you should be able to externalise application config from container images, inject the same ConfigMap and Secret in two or three different ways, and know exactly when changing a ConfigMap will and won't make it into a running pod. Notebooks 02 and 03 are the prerequisites — you should be comfortable writing pod and Deployment manifests, because every consumption pattern in this notebook lives inside `spec.containers`.

## The configuration problem

You finished notebook 03 with a Deployment running an image. That image bakes in a default config. Now you want to:

- Run the same image in *staging* with `LOG_LEVEL=debug` and in *prod* with `LOG_LEVEL=warn`.
- Point the *web* pod at the *api* Service hostname, but keep the URL configurable so you can swap to a different backend later.
- Mount the same TLS certificate into three different services without copying it into three different images.

A few options that **don't** work in production:

**Bake it into the image.** You'd build a separate image per environment. Now `web:staging` and `web:prod` are different images with different test coverage. Promotion is "build a new image" instead of "redeploy the same image with different config" — which means the image you tested in staging isn't the image you ship to prod.

**Pass it on the `docker run` command line.** That's where you came from in the `docker` repo. But pods are launched by the kubelet from a `template` in `etcd`, not by you on the command line. Putting `--env LOG_LEVEL=warn` directly in every Deployment is fine for one variable; with twenty variables, every Deployment YAML becomes a wall of env entries.

**Stash it in a sidecar.** Sometimes the right answer, but heavy for "put a value in an env var."

What you really want is a **first-class Kubernetes object** that holds configuration data, lives in the cluster like every other object, and gets *referenced* from pod templates. Two such objects exist:

- **`ConfigMap`** — for non-sensitive data.
- **`Secret`** — for sensitive data.

They have nearly identical schemas and consumption patterns. The differences are about *handling* (base64 storage, encryption at rest, RBAC defaults) more than about *shape*. Master ConfigMap, and Secret is the same thing plus a security posture.

## ConfigMap — the object

A **ConfigMap** is a key/value store of configuration data, scoped to a namespace, that pods reference by name. The values can be short strings or whole files.

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config
data:
  LOG_LEVEL: "info"
  GREETING: "hello from staging"
  app.properties: |
    server.port=8080
    cache.ttl=300s
    feature.x.enabled=true
```

Two sections under the top level:

- **`data`** — string values. Keys are arbitrary; values are UTF-8 strings (no binary). Most ConfigMaps you'll ever write live here.
- **`binaryData`** — base64-encoded binary blobs. For things that aren't strings (a `.jks` keystore, an image). Less common.

A ConfigMap is just data. It has no `spec` and no `status` — there's nothing for a controller to reconcile. The cluster does nothing with a ConfigMap until a pod *references* it.

### Size limit

A ConfigMap (and a Secret) is capped at **1 MiB** total. This is an `etcd` limit — every Kubernetes object has to fit in one `etcd` value. For larger configuration (a 10 MB lookup table), put it in object storage, a database, or a sidecar — not a ConfigMap.

In [ ]:
# Imperative: from literal values
!kubectl create configmap app-config \
  --from-literal=LOG_LEVEL=info \
  --from-literal=GREETING='hello from staging'
!echo '---'
# Imperative: from a file (the key becomes the filename, the value is the file's contents)
!cat <<'EOF' > /tmp/app.properties
server.port=8080
cache.ttl=300s
feature.x.enabled=true
EOF
!kubectl create configmap app-file --from-file=/tmp/app.properties
!echo '---'
# Declarative: a YAML manifest
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config-yaml
data:
  LOG_LEVEL: warn
  GREETING: hello from prod
EOF
!echo '---'
!kubectl get configmaps
!echo '---'
!kubectl describe configmap app-config

## Three ways to consume a ConfigMap

A ConfigMap on its own does nothing. A pod *uses* it in one (or more) of three ways.

### 1. As individual env vars — `env[].valueFrom.configMapKeyRef`

Pull one key out of one ConfigMap into one env var. Most precise, most explicit.

```yaml
containers:
  - name: app
    image: busybox
    env:
      - name: LOG_LEVEL
        valueFrom:
          configMapKeyRef:
            name: app-config
            key: LOG_LEVEL
```

Use this when you want to rename a key (`LOG_LEVEL` in the ConfigMap, `LOGGING_LEVEL_ROOT` in the container) or only want a subset of a ConfigMap's keys.

### 2. As a bulk env import — `envFrom.configMapRef`

Slurp *every* key/value pair from a ConfigMap into env vars, with the key as the variable name.

```yaml
containers:
  - name: app
    image: busybox
    envFrom:
      - configMapRef:
          name: app-config
        prefix: APP_      # optional namespacing — LOG_LEVEL becomes APP_LOG_LEVEL
```

Use this when the ConfigMap's keys are already your env-var names.

### 3. As files in a volume — `volume.configMap`

Mount the ConfigMap as a directory. Each key becomes a file; the file's contents are the value. Perfect for "the app reads `/etc/app/config.yaml`."

```yaml
spec:
  containers:
    - name: app
      image: busybox
      volumeMounts:
        - name: cfg
          mountPath: /etc/app
  volumes:
    - name: cfg
      configMap:
        name: app-file
        # optional: project only specific keys, rename them, set file mode
        items:
          - key: app.properties
            path: app.properties
            mode: 0644
```

The volume-mount form is the right choice when the application reads a *file* (most JVM apps, anything that uses `--config /path/to/file`).

### Picking between them

| If the app… | Use… |
|---|---|
| reads env vars and you want a few specific ones | `env` with `configMapKeyRef` |
| reads many env vars from a single source | `envFrom` |
| reads a configuration file | volume mount |
| needs config to live-update without a pod restart | volume mount (see next section) |

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: cm-consumer
spec:
  restartPolicy: Never
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c"]
      args:
        - |
          echo "--- env var from configMapKeyRef:"
          echo "LOG_LEVEL=$LOG_LEVEL"
          echo "--- env vars from envFrom (prefixed APP_):"
          env | grep '^APP_' | sort
          echo "--- file mounted from a configMap volume:"
          cat /etc/app/app.properties
          sleep 30
      env:
        - name: LOG_LEVEL
          valueFrom:
            configMapKeyRef:
              name: app-config
              key: LOG_LEVEL
      envFrom:
        - configMapRef:
            name: app-config
          prefix: APP_
      volumeMounts:
        - name: cfg
          mountPath: /etc/app
  volumes:
    - name: cfg
      configMap:
        name: app-file
EOF
!sleep 6
!kubectl logs cm-consumer

## Update propagation — env vars vs volume mounts

The big practical difference between the three consumption patterns is **what happens when you update the ConfigMap**.

### Env vars are baked in

Both `env` (`configMapKeyRef`) and `envFrom` are resolved **once, at pod start**. The kubelet reads the ConfigMap, builds the env block, and `exec`s the container. After that, changing the ConfigMap **does nothing** to running pods. To pick up the new value, you have to restart the pod — typically by triggering a rollout (notebook 03 — bump an annotation on the Deployment's pod template).

### Volume-mounted ConfigMaps live-update

When a ConfigMap is mounted as a volume, the kubelet writes its keys to a small `tmpfs` directory inside the pod. When the ConfigMap changes, the kubelet **rewrites those files** within roughly a minute (the sync period; tunable). The container sees new file contents without restarting.

This is great — your nginx config reloads when you `kubectl apply` an updated ConfigMap. It's also a footgun — your container has to actually *re-read* the file. Many apps load config once at startup; the file might be fresh but the application is still using the old values. The standard fixes:

- **Have the app watch the file** (inotify, periodic re-read).
- **Trigger a rollout on every ConfigMap change** (a common GitOps pattern, often done with a checksum annotation in the pod template).

### The `subPath` gotcha

A volume mounted with `subPath` is **not live-updated**. `subPath` mounts a single file from a volume at a target path (for example, mount only `nginx.conf` into `/etc/nginx/nginx.conf` without clobbering the rest of `/etc/nginx`). It's a separate kernel bind mount and the kubelet doesn't update it.

If you need live updates *and* want to land a single file in a directory you can't replace, mount the whole ConfigMap somewhere harmless (`/etc/configs/`) and have the app read from there, or use a sidecar to copy the file on changes.

## Immutable ConfigMaps (and Secrets)

By default the kubelet watches every ConfigMap a pod mounts, so it can notice changes. On a large cluster, thousands of ConfigMaps × thousands of pods × thousands of nodes means a lot of watches.

Setting `immutable: true` on a ConfigMap tells the kubelet **this never changes — don't bother watching it.** Once you flip the bit, the ConfigMap can only be deleted and recreated, not edited.

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config-v3   # version in the name, since you can't edit it
data:
  LOG_LEVEL: warn
immutable: true
```

The same field exists on Secrets. Use it for "release-versioned" ConfigMaps where the file is born, deployed, and dies with one release — you never patch it.

The typical pattern: ship `app-config-v3` alongside Deployment `web-v3`, then delete the pair when you tear down that version. Tools like Kustomize and Helm can append a hash of the data to the name automatically, so every change effectively creates a new immutable object.

## Secrets — like ConfigMaps but for sensitive data

A **Secret** is the same idea as a ConfigMap, plus a handful of differences that exist because the data is sensitive:

```yaml
apiVersion: v1
kind: Secret
metadata:
  name: db-credentials
type: Opaque
data:
  username: cG9zdGdyZXM=        # "postgres" in base64
  password: c3VwM3JfczNjcjN0    # "sup3r_s3cr3t" in base64
stringData:
  api_token: "raw-string-token"  # convenience: written here, base64'd by the API server
```

What's different from a ConfigMap:

- **Storage.** Secret values in YAML use `data:` (base64-encoded) or `stringData:` (raw — the API server base64-encodes it before storing). ConfigMap doesn't bother with base64 for strings.
- **In-memory mounting.** When a Secret is mounted as a volume, the kubelet puts it on `tmpfs` (RAM, not disk) so it never touches a node's disk.
- **etcd storage.** By default, Secret data sits in `etcd` as base64 — **not encrypted**. Most production clusters enable an `EncryptionConfiguration` so the API server encrypts Secret data before writing it to `etcd`. The CKA expects you to know that this is *optional but recommended*, and that it's the API server, not `etcd` itself, doing the encryption.
- **RBAC defaults.** Many cluster role definitions exclude `secrets` from "read everything in this namespace." Treat Secrets as more sensitive than ConfigMaps by policy, even though the underlying object kind is similar.

### base64 is encoding, not encryption

The single most important fact about Secrets:

> **`base64` is not encryption. It's an encoding. Anyone with `get secret` access can decode it instantly.**

```bash
echo cG9zdGdyZXM= | base64 -d   # -> postgres
```

If you commit a Secret YAML to Git (even base64-encoded), you've committed plaintext. The community-standard solutions are **Sealed Secrets** (encrypts so only the in-cluster controller can decrypt), **External Secrets Operator** (syncs from Vault, AWS Secrets Manager, etc.), and **SOPS** (encrypts the YAML with KMS or PGP). Out of scope for the CKA; standard for production.

## Creating Secrets

The same three creation methods as ConfigMap, plus a few Secret-specific shortcuts.

**From literals**:

```bash
kubectl create secret generic db-credentials \
  --from-literal=username=postgres \
  --from-literal=password='sup3r_s3cr3t'
```

**From files** (handy for binary or large values like a private key):

```bash
kubectl create secret generic tls-ca --from-file=ca.crt=/path/to/ca.crt
```

**From a TLS cert + key pair** (the `kubernetes.io/tls` typed shortcut):

```bash
kubectl create secret tls web-tls --cert=server.crt --key=server.key
```

**From a Docker registry credential** (the `dockerconfigjson` typed shortcut):

```bash
kubectl create secret docker-registry regcred \
  --docker-server=registry.example.com \
  --docker-username=ci-bot \
  --docker-password='<password>' \
  --docker-email=ci@example.com
```

**Declarative**: the YAML shown above, with `data:` or `stringData:`. Use `stringData` while you're authoring; the API server converts it to base64 on write.

### `data` vs `stringData` precedence

If both are set and a key appears in each, `stringData` wins. Useful when patching.

In [ ]:
# Generic Secret from literals
!kubectl create secret generic db-credentials \
  --from-literal=username=postgres \
  --from-literal=password='sup3r_s3cr3t'
!echo '---'
# Inspect — note `data:` is base64, not plaintext
!kubectl get secret db-credentials -o yaml | sed -n '1,15p'
!echo '---'
# Decode (for demonstration — proves base64 is not encryption)
!kubectl get secret db-credentials -o jsonpath='{.data.password}' | base64 -d
!echo

## Consuming Secrets

Identical to ConfigMaps. Three ways.

### As an env var — `secretKeyRef`

```yaml
env:
  - name: DB_PASSWORD
    valueFrom:
      secretKeyRef:
        name: db-credentials
        key: password
```

### As a bulk env import — `envFrom`

```yaml
envFrom:
  - secretRef:
      name: db-credentials
```

Every key in the Secret becomes an env var. As with ConfigMaps, `prefix:` is supported.

### As files in a volume — `volume.secret`

```yaml
spec:
  containers:
    - name: app
      image: busybox
      volumeMounts:
        - name: secret-vol
          mountPath: /etc/secrets
          readOnly: true
  volumes:
    - name: secret-vol
      secret:
        secretName: db-credentials
        defaultMode: 0400   # files readable only by the owner
        items:
          - key: password
            path: db-password
```

Two Secret-specific volume knobs:

- **`defaultMode`** (and `items[].mode`) — file permissions. Default `0644`. For sensitive data, prefer `0400` so only the container's user can read.
- **`tmpfs` backing.** As mentioned above, Secret volumes are RAM-backed. Automatic — you don't ask for it.

Update propagation is the same as ConfigMaps — env vars are baked at pod start, volumes live-refresh.

### Don't `echo` Secrets in container logs

Sounds obvious. Bears repeating. The most common Secret leak in the wild is `echo $DB_PASSWORD` left in an entrypoint script, or a misconfigured Spring Boot that logs every env var on startup. Pods' stdout streams to node disks and from there to your logging pipeline.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: sec-consumer
spec:
  restartPolicy: Never
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c"]
      args:
        - |
          echo "--- env var from secretKeyRef:"
          echo "DB_USER=$DB_USER"
          echo "--- file from secret volume (note 0400 mode):"
          ls -l /etc/secrets/
          echo "--- contents:"
          cat /etc/secrets/username
          echo
          sleep 30
      env:
        - name: DB_USER
          valueFrom:
            secretKeyRef:
              name: db-credentials
              key: username
      volumeMounts:
        - name: cred-vol
          mountPath: /etc/secrets
          readOnly: true
  volumes:
    - name: cred-vol
      secret:
        secretName: db-credentials
        defaultMode: 0400
EOF
!sleep 5
!kubectl logs sec-consumer

## Built-in Secret types

A Secret's `type` field is a hint about *what's inside*. The cluster doesn't enforce the schema for most types — it's documentation that some controllers check for. Common types:

| Type | What's in `data` | Used by |
|---|---|---|
| `Opaque` *(default)* | arbitrary key/value | most user secrets |
| `kubernetes.io/service-account-token` | a JWT for a ServiceAccount | auto-created per SA; injected into pods |
| `kubernetes.io/dockercfg` | a legacy `~/.dockercfg` JSON | image pull (legacy) |
| `kubernetes.io/dockerconfigjson` | a `~/.docker/config.json` blob | image pull (modern) |
| `kubernetes.io/basic-auth` | `username` and `password` keys | by convention |
| `kubernetes.io/ssh-auth` | `ssh-privatekey` key | by convention |
| `kubernetes.io/tls` | `tls.crt` and `tls.key` keys | Ingress, webhook servers |
| `bootstrap.kubernetes.io/token` | `kubeadm` join tokens | cluster bootstrap |

The two you'll create most often:

### TLS secrets

```yaml
apiVersion: v1
kind: Secret
metadata:
  name: web-tls
type: kubernetes.io/tls
data:
  tls.crt: LS0tLS1CRUdJTi...    # base64 of the PEM cert
  tls.key: LS0tLS1CRUdJTi...    # base64 of the PEM key
```

Created most easily with `kubectl create secret tls`. Consumed by Ingress resources (notebook 09) and any pod that needs to serve HTTPS.

### Image pull secrets — covered next.

## Image pull secrets

If your pod runs an image from a **private registry**, the kubelet on the node needs credentials to pull it. That's what `imagePullSecrets` is for.

### Step 1 — create the credential as a Secret

```bash
kubectl create secret docker-registry regcred \
  --docker-server=registry.example.com \
  --docker-username=ci-bot \
  --docker-password='<password>'
```

That creates a Secret of type `kubernetes.io/dockerconfigjson` whose `data` is a base64 of a `~/.docker/config.json` blob.

### Step 2 — reference it from the pod spec

```yaml
spec:
  imagePullSecrets:
    - name: regcred
  containers:
    - name: app
      image: registry.example.com/team/app:1.2.3
```

That's it. The kubelet uses `regcred` to authenticate when pulling the image. The Secret is **only** used for image pull; the running container never sees it as an env var or a file.

### Cluster-wide alternative: attach to the ServiceAccount

If every pod in a namespace needs the same registry credential, set it once on the namespace's default ServiceAccount and skip the per-pod `imagePullSecrets`:

```bash
kubectl patch serviceaccount default \
  -p '{"imagePullSecrets": [{"name": "regcred"}]}'
```

Every pod created in this namespace that *doesn't override the ServiceAccount* now picks up `regcred` automatically. ServiceAccounts land properly in notebook 10 alongside RBAC.

## The downward API — pod metadata as env vars or files

Sometimes the container needs to know *something about itself*: its own pod name, namespace, IP, node, labels, or resource limits. None of that is in a ConfigMap or Secret. It's in the pod's own object, and the **downward API** exposes it.

Two surfaces.

### As env vars

```yaml
containers:
  - name: app
    image: busybox
    env:
      - name: POD_NAME
        valueFrom:
          fieldRef:
            fieldPath: metadata.name
      - name: POD_NAMESPACE
        valueFrom:
          fieldRef:
            fieldPath: metadata.namespace
      - name: POD_IP
        valueFrom:
          fieldRef:
            fieldPath: status.podIP
      - name: NODE_NAME
        valueFrom:
          fieldRef:
            fieldPath: spec.nodeName
      - name: CPU_LIMIT
        valueFrom:
          resourceFieldRef:
            containerName: app
            resource: limits.cpu
            divisor: 1m          # in millicores
```

### As files in a volume

```yaml
volumes:
  - name: podinfo
    downwardAPI:
      items:
        - path: "labels"
          fieldRef:
            fieldPath: metadata.labels
        - path: "annotations"
          fieldRef:
            fieldPath: metadata.annotations
```

Mount that volume and the container reads its labels and annotations from files. Useful when an app expects "configuration" but really wants "tell me which environment I'm in."

### Available fields

`fieldRef` accepts (the most-used set):

- `metadata.name`, `metadata.namespace`, `metadata.uid`
- `metadata.labels`, `metadata.annotations`, `metadata.labels['key']`
- `spec.nodeName`, `spec.serviceAccountName`
- `status.hostIP`, `status.podIP`, `status.podIPs`

`resourceFieldRef` accepts `limits.cpu`, `limits.memory`, `limits.ephemeral-storage`, `requests.*` — useful for sizing JVM heaps, GC threads, and the like against actual pod limits.

We already used the downward API in notebook 04 — the `POD_NAME` env var on the `http-echo` Deployment came from `fieldRef`.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: downward-demo
  labels:
    app: demo
    tier: example
spec:
  restartPolicy: Never
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c"]
      args:
        - |
          echo "I am pod $POD_NAME in namespace $POD_NAMESPACE"
          echo "Running on node $NODE_NAME with IP $POD_IP"
          echo "--- labels from downward API volume:"
          cat /etc/podinfo/labels
          echo
          sleep 30
      env:
        - name: POD_NAME
          valueFrom: { fieldRef: { fieldPath: metadata.name } }
        - name: POD_NAMESPACE
          valueFrom: { fieldRef: { fieldPath: metadata.namespace } }
        - name: POD_IP
          valueFrom: { fieldRef: { fieldPath: status.podIP } }
        - name: NODE_NAME
          valueFrom: { fieldRef: { fieldPath: spec.nodeName } }
      volumeMounts:
        - name: podinfo
          mountPath: /etc/podinfo
  volumes:
    - name: podinfo
      downwardAPI:
        items:
          - path: labels
            fieldRef: { fieldPath: metadata.labels }
EOF
!sleep 5
!kubectl logs downward-demo

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 06:

```bash
kubectl delete pod cm-consumer sec-consumer downward-demo
kubectl delete configmap app-config app-file app-config-yaml
kubectl delete secret db-credentials
```

Or, more bluntly:

```bash
kubectl delete pod,configmap,secret --all
```